# Time Series & Datetime Analysis (5+ Years Interview Guide)
Exhaustive senior guide to datetime parsing, .dt accessors, time zone localization, frequency resampling, business day offsets, and irregular windowing.

### Key 5-Year Interview Concepts Covered:
- **Parsing Robustness**: Parsing mixed formats with `pd.to_datetime(format='mixed', errors='coerce')`.
- **Vectorized `.dt` Accessors**: Extracting calendar components (`.dt.year`, `.dt.day_name()`, `.dt.is_month_end`).
- **Timezone Localization & Conversion**: Handling UTC, EST, and DST transitions using `.dt.tz_localize()` and `.dt.tz_convert()`.
- **Resampling vs Groupby**: Downsampling/upsampling frequencies (`.resample('ME')`, `.resample('W-MON')`) with aggregation.
- **Business Offsets & Date Ranges**: Generating trading day grids with `pd.bdate_range()` and `pd.offsets.BDay()`.

This interactive notebook is fully customized using the Fintech dataset `data/raw_transactions.csv`.

In [ ]:
# Setup imports & load dataset
import pandas as pd
import numpy as np
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path, na_values=['Nan', ''])
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(2)

## Section 1: Datetime Ingestion & Parsing

### Robust Datetime Parsing (`pd.to_datetime`)
**Explanation**: Parsing dates into `datetime64[ns]` timestamp objects. Passing `format='mixed'` parses heterogeneous date formats (e.g. '07-06-2025' and '2025-07-26 06:22:30'), while `errors='coerce'` converts unparseable junk strings to `NaT` (Not-a-Time) rather than crashing the ingestion pipeline.

**Syntax**: `pd.to_datetime(series, format='mixed', errors='coerce')`

In [ ]:
df['parsed_date'] = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
print(df[['transaction_date', 'parsed_date']].head(4))
print('Null timestamps after parsing:', df['parsed_date'].isna().sum())

### Epoch Timestamp Conversions
**Explanation**: In high-frequency trading and fintech messaging queues, timestamps arrive as integer milliseconds or seconds since Unix Epoch (1970-01-01). Passing `unit='s'` or `unit='ms'` reconstructs standard datetime representations instantly in C without string parsing overhead.

**Syntax**: `pd.to_datetime(numeric_series, unit='s')`

In [ ]:
sample_epochs = pd.Series([1751785350, 1751871750, 1751958150])
converted_dates = pd.to_datetime(sample_epochs, unit='s')
print(converted_dates)

## Section 2: Vectorized `.dt` Accessor Operations

### Calendar & Period Component Extraction
**Explanation**: The `.dt` accessor provides vectorized methods running at C-speed to extract calendar attributes: `.dt.year`, `.dt.month`, `.dt.quarter`, `.dt.day_name()`, and boolean flags like `.dt.is_month_end`. These are essential for building seasonal feature stores and financial reporting tables.

**Syntax**: `df['date'].dt.year` / `df['date'].dt.day_name()` / `df['date'].dt.is_month_end`

In [ ]:
df['year'] = df['parsed_date'].dt.year
df['month'] = df['parsed_date'].dt.month
df['day_name'] = df['parsed_date'].dt.day_name()
df['is_month_end'] = df['parsed_date'].dt.is_month_end
print(df[['parsed_date', 'year', 'month', 'day_name', 'is_month_end']].head(4))

### Time Zone Localization & Conversion
**Explanation**: Timestamps parsed from raw logs are usually 'timezone-naive' (no offset). `.dt.tz_localize('UTC')` binds timezone metadata without changing the time. `.dt.tz_convert('America/New_York')` then translates UTC timestamps into local market trading time, correctly adjusting for Daylight Saving Time (DST).

**Syntax**: `series.dt.tz_localize('UTC').dt.tz_convert('America/New_York')`

In [ ]:
valid_dates = df['parsed_date'].dropna().head(3)
tz_aware_utc = valid_dates.dt.tz_localize('UTC')
tz_aware_ny = tz_aware_utc.dt.tz_convert('America/New_York')
print('UTC:\n', tz_aware_utc)
print('\nNew York (EST/EDT):\n', tz_aware_ny)

## Section 3: Frequency Resampling & Business Offsets

### Time-Series Resampling (`.resample()`)
**Explanation**: Resampling is time-based grouping for DataFrames with a `DatetimeIndex`. In Pandas 2.2+, `'ME'` (Month End), `'W-MON'` (Weekly Monday), and `'QE'` (Quarter End) downsample high-frequency transactions into aggregate financial metrics. Unlike `groupby()`, `resample()` includes periods with zero transactions by default.

**Syntax**: `df.set_index('date').resample('ME')['amount'].agg(['sum', 'count'])`

In [ ]:
ts_df = df.dropna(subset=['parsed_date']).set_index('parsed_date').sort_index()
monthly_summary = ts_df.resample('ME')['transaction_amount'].agg(Total_Volume='sum', Tx_Count='count', Avg_Amount='mean')
print(monthly_summary.head(6))

### Business Day Ranges & Financial Offsets
**Explanation**: `pd.bdate_range(start, end)` creates a DatetimeIndex excluding weekends. `pd.offsets.BDay()` and `pd.offsets.MonthEnd()` allow computing contractual settlement dates, billing cycles, and trade clearing deadlines.

**Syntax**: `pd.bdate_range(start='2025-01-01', periods=5)` / `date + pd.offsets.BDay(2)`

In [ ]:
settlement_grid = pd.bdate_range(start='2025-07-01', periods=5)
print('Business Days Grid:', settlement_grid)
print('Month End Offset:', settlement_grid[0] + pd.offsets.MonthEnd(1))

### Time Periods vs Timestamps (`to_period()`)
**Explanation**: A `Timestamp` represents a single point in time (e.g. 2025-07-26 06:22:30), whereas a `Period` represents an entire span or interval (e.g. Month '2025-07' or Quarter '2025Q3'). `.to_period('M')` converts timestamps to period indices for clean month-over-month financial accounting comparisons.

**Syntax**: `df['date'].dt.to_period('M')` / `period_series.dt.to_timestamp()`

In [ ]:
period_series = df['parsed_date'].dropna().dt.to_period('M')
print('Period Representation:\n', period_series.head(3))

## Section: Senior Fintech Interview Questions (5+ Years Experience)

### Q1: Month-over-Month Transaction Volume & Growth Rate
**Explanation**: Resample transactions by Month-End (`'ME'`), calculate total volume, and use `.pct_change()` to compute month-over-month percentage growth in revenue.

**Syntax**: `df.set_index('date').resample('ME')['amount'].sum().pct_change() * 100`

In [ ]:
ts_df = df.dropna(subset=['parsed_date']).set_index('parsed_date').sort_index()
mom_growth = ts_df.resample('ME')['transaction_amount'].sum().to_frame(name='Monthly_Volume')
mom_growth['MoM_Growth_%'] = mom_growth['Monthly_Volume'].pct_change() * 100
print(mom_growth.head(6))

### Q2: Weekend vs Weekday Transaction Volume Analysis
**Explanation**: Extract day names using `.dt.day_name()`, create a boolean flag for weekends, and compare average transaction amounts between weekdays and weekends.

**Syntax**: `df['parsed_date'].dt.dayofweek >= 5`

In [ ]:
clean_df = df.dropna(subset=['parsed_date']).copy()
clean_df['is_weekend'] = clean_df['parsed_date'].dt.dayofweek >= 5
weekend_comparison = clean_df.groupby('is_weekend')['transaction_amount'].agg(Avg_Amount='mean', Total_Count='count')
weekend_comparison.index = ['Weekday', 'Weekend']
print(weekend_comparison)